In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub gdown')
    print("Setup complete!")


In [2]:
input_file = 'datasets/finetuning/train.csv'
benchmark_dir = 'datasets/preprocessed'
output_dir = 'models/finetuned/GlotLID_v3'


In [3]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
import fasttext
from huggingface_hub import hf_hub_download

MODEL_NAME = "GlotLID v3"
MODEL_ID = "glotlid_v3"


In [4]:
print("Downloading GlotLID v3 base model from Hugging Face (~1.6GB)...")
base_model_path = hf_hub_download(repo_id="cis-lmu/glotlid", filename="model.bin")
base_model_dir = os.path.dirname(base_model_path)

print(f"Loading base GlotLID v3 model from {base_model_path}...")
base_model = fasttext.load_model(base_model_path)

# Extract pretrained word vectors for transfer learning / fine-tuning
vec_file_path = os.path.join(base_model_dir, "glotlid_v3.vec")
if not os.path.exists(vec_file_path):
    print("Extracting pre-trained word vectors to .vec format...")
    words = base_model.get_words()
    dim = base_model.get_dimension()
    with open(vec_file_path, "w", encoding="utf-8") as f:
        f.write(f"{len(words)} {dim}\n")
        for word in words:
            v_str = " ".join(map(str, base_model.get_word_vector(word)))
            f.write(f"{word} {v_str}\n")
    print(f"Extracted {len(words)} vectors into {vec_file_path}")


Loading base GlotLID v3 model from C:\Users\USER\.cache\huggingface\hub\models--cis-lmu--glotlid\snapshots\85cd6716494360367b75f642b5bc78667605d0b4\model.bin...


In [11]:
import fasttext

base_model = fasttext.load_model(base_model_path)

print("Number of labels:", len(base_model.get_labels()))
print("Dimension:", base_model.get_dimension())
print("Input matrix:", base_model.get_input_matrix().shape)
print("Output matrix:", base_model.get_output_matrix().shape)

print("\nModel arguments:")
print(base_model.f.getArgs())

labels = base_model.get_labels()

print("\nPossible Sinhala/Pali/Sanskrit labels:")
for label in labels:
    l = label.lower()
    if any(x in l for x in [
        "sin", "sinhala",
        "pli", "pali",
        "san", "sanskrit"
    ]):
        print(label)

Number of labels: 2102
Dimension: 256
Input matrix: (1634361, 256)
Output matrix: (2102, 256)

Model arguments:

Possible Sinhala/Pali/Sanskrit labels:
__label__sin_Sinh
__label__san_Deva
__label__und_Sinh
__label__und_Sind
__label__san_Latn


In [12]:
args = base_model.f.getArgs()

print("Loss:", args.loss)
print("Model:", args.model)
print("Dimension:", args.dim)
print("Epoch:", args.epoch)
print("Learning rate:", args.lr)

Loss: loss_name.softmax
Model: model_name.supervised
Dimension: 256
Epoch: 1
Learning rate: 0.05


In [5]:
print(f"Loading finetuning dataset from {input_file}...")
df = pd.read_csv(input_file)
print(f"Loaded {len(df)} rows.")

def format_text(text):
    return str(text).replace("\n", " ").strip()

df["clean_text"] = df["text"].apply(format_text)
df["ft_line"] = "__label__" + df["label"].astype(str) + " " + df["clean_text"]

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])

os.makedirs(output_dir, exist_ok=True)
train_ft_file = os.path.join(output_dir, "train_formatted.txt")
val_ft_file = os.path.join(output_dir, "val_formatted.txt")

with open(train_ft_file, "w", encoding="utf-8") as f:
    f.write("\n".join(train_df["ft_line"].tolist()) + "\n")

with open(val_ft_file, "w", encoding="utf-8") as f:
    f.write("\n".join(val_df["ft_line"].tolist()) + "\n")

print(f"Saved {len(train_df)} training samples to {train_ft_file}")
print(f"Saved {len(val_df)} validation samples to {val_ft_file}")


Loading finetuning dataset from datasets/finetuning/train.csv...
Loaded 60285 rows.
Saved 54256 training samples to models/finetuned/GlotLID_v3\train_formatted.txt
Saved 6029 validation samples to models/finetuned/GlotLID_v3\val_formatted.txt


In [6]:
print(f"Starting fine-tuning for {MODEL_NAME}...")
finetuned_model = fasttext.train_supervised(
    input=train_ft_file,
    pretrainedVectors=vec_file_path,
    dim=base_model.get_dimension(),
    epoch=25,
    lr=0.5,
    wordNgrams=2,
    loss="softmax"
)

save_model_path = os.path.join(output_dir, f"{MODEL_ID}_finetuned.bin")
finetuned_model.save_model(save_model_path)
print(f"Fine-tuned model successfully saved to {save_model_path}")


Starting fine-tuning for GlotLID v3...
Fine-tuned model successfully saved to models/finetuned/GlotLID_v3\glotlid_v3_finetuned.bin


In [7]:
val_texts = val_df["clean_text"].tolist()
val_true = val_df["label"].astype(str).tolist()

print(f"Evaluating fine-tuned {MODEL_NAME} on {len(val_texts)} validation samples...")
preds, _ = finetuned_model.predict(val_texts, k=1)
val_pred = [p[0].replace("__label__", "") for p in preds]

acc = accuracy_score(val_true, val_pred)
macro_f1 = f1_score(val_true, val_pred, average="macro")

print("\n" + "=" * 48)
print(f"VALIDATION FINE-TUNING RESULTS ({MODEL_NAME})")
print("=" * 48)
print(f"Accuracy:  {acc * 100:.2f}%")
print(f"Macro F1:  {macro_f1 * 100:.2f}%")
print("=" * 48)
print("\nPer-language breakdown:\n")
print(classification_report(val_true, val_pred, digits=4, zero_division=0))


Evaluating fine-tuned GlotLID v3 on 6029 validation samples...

VALIDATION FINE-TUNING RESULTS (GlotLID v3)
Accuracy:  98.96%
Macro F1:  98.85%

Per-language breakdown:

              precision    recall  f1-score   support

        pali     0.9803    0.9932    0.9867      2349
    sanskrit     0.9930    0.9773    0.9851      1012
     sinhala     0.9966    0.9910    0.9938      2668

    accuracy                         0.9896      6029
   macro avg     0.9899    0.9872    0.9885      6029
weighted avg     0.9896    0.9896    0.9896      6029



In [8]:
print(f"Evaluating fine-tuned {MODEL_NAME} on Sinhala script target languages across benchmark datasets in {benchmark_dir}...")

TARGET_LANGUAGES = ["sinhala", "pali", "sanskrit"]

def map_benchmark_label(row):
    lbl = row.get("label")
    src = row.get("source")
    if lbl in ["sin", "sin_Sinh", "sinhala", "si"]:
        return "sinhala"
    if lbl in ["pli", "pli_Sinh", "pali", "pi"]:
        return "pali"
    if lbl in ["san_Sinh", "sanskrit"] or (lbl == "san" and src in ["DCS", "SansinNT", "SiDiaC-v2"]):
        return "sanskrit"
    return None

def load_benchmark_dataset(file_path):
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            mapped_label = map_benchmark_label(row)
            if mapped_label:
                row["target_label"] = mapped_label
                records.append(row)
    return pd.DataFrame(records)

benchmark_files = sorted(glob.glob(os.path.join(benchmark_dir, "*.jsonl")))
if not benchmark_files:
    print(f"No benchmark datasets found in {benchmark_dir}.")
else:
    results_dir = os.path.join("datasets", "benchmark_results")
    os.makedirs(results_dir, exist_ok=True)
    
    for file_path in benchmark_files:
        dataset_name = os.path.splitext(os.path.basename(file_path))[0]
        df_bench = load_benchmark_dataset(file_path)
        if df_bench.empty:
            print(f"No matching target languages found in {dataset_name}.")
            continue
        
        texts = df_bench["text"].apply(format_text).tolist()
        print(f"\nEvaluating {len(texts)} target language samples from {dataset_name}...")
        preds, _ = finetuned_model.predict(texts, k=1)
        
        results = df_bench[["text", "label", "source"]].copy()
        results["true_label"] = df_bench["target_label"]
        results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]
        
        acc_b = accuracy_score(results["true_label"], results["predicted_label"])
        macro_f1_b = f1_score(
            results["true_label"], results["predicted_label"],
            average="macro", labels=TARGET_LANGUAGES, zero_division=0
        )
        
        print("=" * 65)
        print(f"BENCHMARK RESULTS ({MODEL_NAME} Finetuned on train.csv - Evaluated on {dataset_name})")
        print("=" * 65)
        print(f"Accuracy:  {acc_b * 100:.2f}%")
        print(f"Macro F1:  {macro_f1_b * 100:.2f}%")
        print("=" * 65)
        print("\nPer-language breakdown (F1 scores & metrics for target languages):\n")
        print(classification_report(
            results["true_label"], results["predicted_label"],
            labels=TARGET_LANGUAGES, digits=4, zero_division=0
        ))
        
        out_csv = os.path.join(results_dir, f"{MODEL_ID}_finetuned_{dataset_name}.csv")
        results.to_csv(out_csv, index=False)
        print(f"Saved benchmark predictions to {out_csv}")


Evaluating fine-tuned GlotLID v3 on Sinhala script target languages across benchmark datasets in datasets/preprocessed...

Evaluating 7047 target language samples from commonlid...
BENCHMARK RESULTS (GlotLID v3 Finetuned on train.csv - Evaluated on commonlid)
Accuracy:  98.94%
Macro F1:  98.80%

Per-language breakdown (F1 scores & metrics for target languages):

              precision    recall  f1-score   support

     sinhala     0.9922    0.9959    0.9941      2693
        pali     0.9852    0.9921    0.9886      3027
    sanskrit     0.9931    0.9699    0.9813      1327

    accuracy                         0.9894      7047
   macro avg     0.9902    0.9859    0.9880      7047
weighted avg     0.9894    0.9894    0.9893      7047

Saved benchmark predictions to datasets\benchmark_results\glotlid_v3_finetuned_commonlid.csv

Evaluating 7047 target language samples from flores_plus...
BENCHMARK RESULTS (GlotLID v3 Finetuned on train.csv - Evaluated on flores_plus)
Accuracy:  98.94%
M

In [9]:
print(f"Evaluating fine-tuned {MODEL_NAME} across ALL benchmark languages in {benchmark_dir}...")

ALL_BENCHMARK_LANGUAGES = [
    "sinhala", "pali", "sanskrit", "sanskrit_deva", "english", "tamil",
    "hindi", "bengali", "arabic", "french", "german"
]

LABEL_MAPPING_ALL = {
    "sin": "sinhala", "sin_Sinh": "sinhala", "sinhala": "sinhala", "si": "sinhala",
    "pli": "pali", "pli_Sinh": "pali", "pli_Latn": "pali", "pali": "pali", "pi": "pali",
    "san_Sinh": "sanskrit",
    "san_Deva": "sanskrit_deva", "sa": "sanskrit_deva",
    "eng": "english", "eng_Latn": "english", "english": "english", "en": "english",
    "tam": "tamil", "tam_Taml": "tamil", "tamil": "tamil", "ta": "tamil",
    "hin": "hindi", "hin_Deva": "hindi", "hindi": "hindi", "hi": "hindi",
    "ben": "bengali", "ben_Beng": "bengali", "bengali": "bengali", "bn": "bengali",
    "arb": "arabic", "arb_Arab": "arabic", "arabic": "arabic", "ar": "arabic",
    "fra": "french", "fra_Latn": "french", "french": "french", "fr": "french",
    "deu": "german", "deu_Latn": "german", "german": "german", "de": "german"
}

def map_all_label(row):
    lbl = row.get("label")
    src = row.get("source")
    if lbl == "san":
        if src in ["DCS", "SansinNT", "SiDiaC-v2"]:
            return "sanskrit"
        else:
            return "sanskrit_deva"
    return LABEL_MAPPING_ALL.get(lbl)

def load_all_languages_dataset(file_path):
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            mapped_label = map_all_label(row)
            if mapped_label:
                row["target_label"] = mapped_label
                records.append(row)
    return pd.DataFrame(records)

benchmark_files = sorted(glob.glob(os.path.join(benchmark_dir, "*.jsonl")))
if not benchmark_files:
    print(f"No benchmark datasets found in {benchmark_dir}.")
else:
    results_dir = os.path.join("datasets", "benchmark_results")
    os.makedirs(results_dir, exist_ok=True)
    
    for file_path in benchmark_files:
        dataset_name = os.path.splitext(os.path.basename(file_path))[0]
        df_all = load_all_languages_dataset(file_path)
        if df_all.empty:
            print(f"No matching languages found in {dataset_name}.")
            continue
        
        texts = df_all["text"].apply(format_text).tolist()
        print(f"\nEvaluating {len(texts)} samples across ALL benchmark languages from {dataset_name}...")
        preds, _ = finetuned_model.predict(texts, k=1)
        
        results = df_all[["text", "label", "source"]].copy()
        results["true_label"] = df_all["target_label"]
        results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]
        
        acc_all = accuracy_score(results["true_label"], results["predicted_label"])
        macro_f1_all = f1_score(
            results["true_label"], results["predicted_label"],
            average="macro", labels=ALL_BENCHMARK_LANGUAGES, zero_division=0
        )
        
        print("=" * 65)
        print(f"ALL LANGUAGES BENCHMARK RESULTS ({MODEL_NAME} Finetuned on train.csv - Evaluated on {dataset_name})")
        print("=" * 65)
        print(f"Accuracy:  {acc_all * 100:.2f}%")
        print(f"Macro F1:  {macro_f1_all * 100:.2f}%")
        print("=" * 65)
        print("\nPer-language breakdown (All Benchmark Languages):\n")
        print(classification_report(
            results["true_label"], results["predicted_label"],
            labels=ALL_BENCHMARK_LANGUAGES, digits=4, zero_division=0
        ))
        
        out_csv = os.path.join(results_dir, f"{MODEL_ID}_finetuned_all_langs_{dataset_name}.csv")
        results.to_csv(out_csv, index=False)
        print(f"Saved all-languages benchmark predictions to {out_csv}")


Evaluating fine-tuned GlotLID v3 across ALL benchmark languages in datasets/preprocessed...

Evaluating 77974 samples across ALL benchmark languages from commonlid...
ALL LANGUAGES BENCHMARK RESULTS (GlotLID v3 Finetuned on train.csv - Evaluated on commonlid)
Accuracy:  8.94%
Macro F1:  6.40%

Per-language breakdown (All Benchmark Languages):

               precision    recall  f1-score   support

      sinhala     0.0663    0.9959    0.1243      2693
         pali     0.0922    0.9921    0.1687      3027
     sanskrit     0.2612    0.9699    0.4116      1327
sanskrit_deva     0.0000    0.0000    0.0000       895
      english     0.0000    0.0000    0.0000     27461
        tamil     0.0000    0.0000    0.0000        81
        hindi     0.0000    0.0000    0.0000      3666
      bengali     0.0000    0.0000    0.0000      1886
       arabic     0.0000    0.0000    0.0000     26152
       french     0.0000    0.0000    0.0000      3233
       german     0.0000    0.0000    0.0000    

In [10]:
print("\n" + "=" * 50)
print(f"FINE-TUNING & BENCHMARKING COMPLETE FOR {MODEL_NAME}")
print(f"Fine-tuned model saved to: {save_model_path}")
print("=" * 50)



FINE-TUNING & BENCHMARKING COMPLETE FOR GlotLID v3
Fine-tuned model saved to: models/finetuned/GlotLID_v3\glotlid_v3_finetuned.bin


In [13]:
# ============================================================
# GLOTLID v3
# ZERO-SHOT + FINETUNED
# ALL-LANGUAGE BENCHMARK EVALUATION
#
# Evaluates on:
#   - CommonLID
#   - FLORES+
#   - WiLI-2018
#
# Spreadsheet languages:
# Sinhala-Sinh, Pali-Sinh, Sanskrit-Sinh, Sanskrit-Deva,
# English-Latn, Tamil-Taml, Hindi-Deva, Bengali-Beng,
# Arabic-Arab, French-Latn, German-Latn
# ============================================================

import os
import json
import glob
import gc
import fasttext
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)


# ============================================================
# 1. PATHS
# ============================================================

# These paths assume the notebook is running from:
# data_pipeline/

ZERO_SHOT_MODEL_PATH = (
    "models/finetuned/GlotLID_v3/"
    "glotlid_v3_original.bin"
)

FINETUNED_MODEL_PATH = (
    "models/finetuned/GlotLID_v3/"
    "glotlid_v3_2104_ft_e5_lr0005.bin"
)

BENCHMARK_INPUT_DIR = "datasets/preprocessed"

BENCHMARK_OUTPUT_DIR = "datasets/benchmark_results"

os.makedirs(
    BENCHMARK_OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 2. LANGUAGES USED IN YOUR SPREADSHEET
# ============================================================

ALL_BENCHMARK_LANGUAGES = [
    "sinhala",
    "pali",
    "sanskrit",
    "sanskrit_deva",
    "english",
    "tamil",
    "hindi",
    "bengali",
    "arabic",
    "french",
    "german",
]


# Pretty names matching your spreadsheet
DISPLAY_NAMES = {
    "sinhala": "Sinhala-Sinh",
    "pali": "Pali-Sinh",
    "sanskrit": "Sanskrit-Sinh",
    "sanskrit_deva": "Sanskrit-Deva",
    "english": "English-Latn",
    "tamil": "Tamil-Taml",
    "hindi": "Hindi-Deva",
    "bengali": "Bengali-Beng",
    "arabic": "Arabic-Arab",
    "french": "French-Latn",
    "german": "German-Latn",
}


# ============================================================
# 3. BENCHMARK TRUE-LABEL MAPPING
# ============================================================

LABEL_MAPPING_ALL = {

    # Sinhala-Sinh
    "sin": "sinhala",
    "sin_Sinh": "sinhala",
    "si": "sinhala",
    "sinhala": "sinhala",

    # Pali-Sinh
    "pli": "pali",
    "pli_Sinh": "pali",
    "pali": "pali",
    "pi": "pali",

    # Sanskrit-Deva explicit aliases
    "san_Deva": "sanskrit_deva",
    "sa": "sanskrit_deva",

    # English
    "eng": "english",
    "eng_Latn": "english",
    "en": "english",
    "english": "english",

    # Tamil
    "tam": "tamil",
    "tam_Taml": "tamil",
    "ta": "tamil",
    "tamil": "tamil",

    # Hindi
    "hin": "hindi",
    "hin_Deva": "hindi",
    "hi": "hindi",
    "hindi": "hindi",

    # Bengali
    "ben": "bengali",
    "ben_Beng": "bengali",
    "bn": "bengali",
    "bengali": "bengali",

    # Arabic
    "arb": "arabic",
    "arb_Arab": "arabic",
    "ar": "arabic",
    "arabic": "arabic",

    # French
    "fra": "french",
    "fra_Latn": "french",
    "fr": "french",
    "french": "french",

    # German
    "deu": "german",
    "deu_Latn": "german",
    "de": "german",
    "german": "german",
}


# Project sources containing Sanskrit written in Sinhala script
PROJECT_SANSKRIT_SOURCES = {
    "DCS",
    "SansinNT",
    "SiDiaC-v2",
}


def map_true_label(row):
    """
    Convert benchmark labels into the labels used by our spreadsheet.
    Most importantly, keep Sanskrit-Sinh and Sanskrit-Deva separate.
    """

    label = str(
        row.get("label", "")
    ).strip()

    source = str(
        row.get("source", "")
    ).strip()

    # Explicit Sinhala-script Sanskrit
    if label == "san_Sinh":
        return "sanskrit"

    # Explicit Devanagari Sanskrit
    if label == "san_Deva":
        return "sanskrit_deva"

    # Ambiguous raw Sanskrit label
    if label in ["san", "sanskrit"]:

        if source in PROJECT_SANSKRIT_SOURCES:
            return "sanskrit"

        return "sanskrit_deva"

    return LABEL_MAPPING_ALL.get(label)


# ============================================================
# 4. GLOTLID MODEL-LABEL MAPPING
# ============================================================

# Both models use the same mapping here.
#
# The ORIGINAL GlotLID model does NOT contain:
#   __label__pli_Sinh
#   __label__san_Sinh
#
# Therefore zero-shot F1 for those classes may be 0,
# which is valid if benchmark support > 0.
#
# The fine-tuned 2104-label model DOES contain them.

MODEL_LABEL_TO_NAME = {

    # Target languages
    "__label__sin_Sinh": "sinhala",
    "__label__pli_Sinh": "pali",
    "__label__san_Sinh": "sanskrit",

    # Sanskrit in Devanagari
    "__label__san_Deva": "sanskrit_deva",

    # Existing languages
    "__label__eng_Latn": "english",
    "__label__tam_Taml": "tamil",
    "__label__hin_Deva": "hindi",
    "__label__ben_Beng": "bengali",
    "__label__arb_Arab": "arabic",
    "__label__fra_Latn": "french",
    "__label__deu_Latn": "german",
}


# ============================================================
# 5. LOAD BENCHMARK DATA
# ============================================================

def load_benchmark(file_path):

    records = []

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            row = json.loads(line)

            mapped = map_true_label(row)

            if mapped in ALL_BENCHMARK_LANGUAGES:

                row["target_label"] = mapped

                records.append(row)

    return pd.DataFrame(records)


benchmark_files = sorted(
    glob.glob(
        os.path.join(
            BENCHMARK_INPUT_DIR,
            "*.jsonl"
        )
    )
)


print("=" * 80)
print("BENCHMARK FILES")
print("=" * 80)

for file_path in benchmark_files:
    print(os.path.basename(file_path))


# Read benchmark files once
benchmark_datasets = {}

for file_path in benchmark_files:

    dataset_name = os.path.splitext(
        os.path.basename(file_path)
    )[0]

    df = load_benchmark(file_path)

    if not df.empty:
        benchmark_datasets[dataset_name] = df


print("\nLoaded benchmark rows:")

for name, df in benchmark_datasets.items():
    print(f"{name}: {len(df)} rows")


# ============================================================
# 6. PREDICTION FUNCTION
# ============================================================

def clean_text(text):

    return (
        str(text)
        .replace("\n", " ")
        .replace("\r", " ")
        .strip()
    )


def predict_texts(
    model,
    texts,
    batch_size=10000
):
    """
    Predict in batches to avoid using too much RAM.
    """

    all_raw_predictions = []
    all_mapped_predictions = []

    total = len(texts)

    for start in range(
        0,
        total,
        batch_size
    ):

        end = min(
            start + batch_size,
            total
        )

        batch = [
            clean_text(text)
            for text in texts[start:end]
        ]

        predictions, _ = model.predict(
            batch,
            k=1
        )

        for prediction in predictions:

            raw_label = prediction[0]

            mapped_label = (
                MODEL_LABEL_TO_NAME.get(
                    raw_label,
                    "other"
                )
            )

            all_raw_predictions.append(
                raw_label
            )

            all_mapped_predictions.append(
                mapped_label
            )

    return (
        all_raw_predictions,
        all_mapped_predictions
    )


# ============================================================
# 7. EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model_path,
    model_title,
    output_prefix
):

    print("\n")
    print("#" * 100)
    print(model_title)
    print("#" * 100)

    print("\nLoading model:")
    print(model_path)

    model = fasttext.load_model(
        model_path
    )

    model_labels = set(
        model.get_labels()
    )

    print(
        "Number of model labels:",
        len(model_labels)
    )

    # Useful check for target classes
    print("\nTarget label support in model:")

    for label in [
        "__label__sin_Sinh",
        "__label__pli_Sinh",
        "__label__san_Sinh",
        "__label__san_Deva",
    ]:

        print(
            f"{label:25s}",
            label in model_labels
        )


    overall_summary = []
    per_language_summary = []


    # --------------------------------------------------------
    # Evaluate each benchmark
    # --------------------------------------------------------

    for dataset_name, df_all in benchmark_datasets.items():

        print("\n")
        print("=" * 80)

        print(
            f"{model_title} - "
            f"{dataset_name}"
        )

        print("=" * 80)

        print(
            "Samples:",
            len(df_all)
        )


        # True support
        print("\nTrue-label counts:")

        counts = (
            df_all["target_label"]
            .value_counts()
            .reindex(
                ALL_BENCHMARK_LANGUAGES,
                fill_value=0
            )
        )

        for language, count in counts.items():

            print(
                f"{DISPLAY_NAMES[language]:18s}: "
                f"{count}"
            )


        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        results = df_all.copy()

        raw_predictions, mapped_predictions = (
            predict_texts(
                model,
                results["text"].tolist()
            )
        )

        results["raw_prediction"] = (
            raw_predictions
        )

        results["predicted_label"] = (
            mapped_predictions
        )


        y_true = (
            results["target_label"]
        )

        y_pred = (
            results["predicted_label"]
        )


        # ----------------------------------------------------
        # Overall metrics
        # ----------------------------------------------------

        accuracy = accuracy_score(
            y_true,
            y_pred
        )

        macro_f1 = f1_score(
            y_true,
            y_pred,
            labels=ALL_BENCHMARK_LANGUAGES,
            average="macro",
            zero_division=0
        )


        print("\n" + "-" * 80)

        print(
            f"Accuracy : {accuracy:.4f}"
        )

        print(
            f"Macro F1 : {macro_f1:.4f}"
        )

        print("-" * 80)


        # ----------------------------------------------------
        # Per-language report
        # ----------------------------------------------------

        report = classification_report(
            y_true,
            y_pred,
            labels=ALL_BENCHMARK_LANGUAGES,
            output_dict=True,
            zero_division=0
        )


        print(
            "\nPER-LANGUAGE F1 "
            "(values for spreadsheet)\n"
        )


        for language in ALL_BENCHMARK_LANGUAGES:

            support = int(
                report[language]["support"]
            )

            f1 = float(
                report[language]["f1-score"]
            )


            if support == 0:

                print(
                    f"{DISPLAY_NAMES[language]:18s}: "
                    f"N/A"
                )

                value = None

            else:

                print(
                    f"{DISPLAY_NAMES[language]:18s}: "
                    f"{f1:.4f}"
                )

                value = f1


            per_language_summary.append(
                {
                    "dataset": dataset_name,
                    "language": language,
                    "f1": value,
                    "support": support,
                }
            )


        # ----------------------------------------------------
        # Full report
        # ----------------------------------------------------

        print(
            "\nFull classification report:\n"
        )

        print(
            classification_report(
                y_true,
                y_pred,
                labels=ALL_BENCHMARK_LANGUAGES,
                target_names=[
                    DISPLAY_NAMES[x]
                    for x in ALL_BENCHMARK_LANGUAGES
                ],
                digits=4,
                zero_division=0
            )
        )


        # ----------------------------------------------------
        # Save predictions
        # ----------------------------------------------------

        output_csv = os.path.join(
            BENCHMARK_OUTPUT_DIR,
            f"{output_prefix}_{dataset_name}.csv"
        )

        results.to_csv(
            output_csv,
            index=False
        )

        print(
            "Saved predictions:",
            output_csv
        )


        overall_summary.append(
            {
                "dataset": dataset_name,
                "rows": len(results),
                "accuracy": accuracy,
                "macro_f1": macro_f1,
            }
        )


    # --------------------------------------------------------
    # Remove big model from RAM
    # --------------------------------------------------------

    del model

    gc.collect()


    # ========================================================
    # CREATE SPREADSHEET TABLE
    # ========================================================

    per_language_df = pd.DataFrame(
        per_language_summary
    )


    spreadsheet_table = (
        per_language_df
        .pivot(
            index="dataset",
            columns="language",
            values="f1"
        )
        .reindex(
            columns=ALL_BENCHMARK_LANGUAGES
        )
    )


    spreadsheet_table.columns = [
        DISPLAY_NAMES[column]
        for column in spreadsheet_table.columns
    ]


    spreadsheet_table = (
        spreadsheet_table
        .round(4)
    )


    overall_df = pd.DataFrame(
        overall_summary
    )


    return (
        spreadsheet_table,
        overall_df
    )


# ============================================================
# 8. ZERO-SHOT GLOTLID
# ============================================================

zero_shot_table, zero_shot_overall = (
    evaluate_model(
        model_path=ZERO_SHOT_MODEL_PATH,
        model_title="GLOTLID v3 ZERO-SHOT",
        output_prefix="glotlid_v3_zero_shot"
    )
)


# ============================================================
# 9. FINE-TUNED GLOTLID
# ============================================================

finetuned_table, finetuned_overall = (
    evaluate_model(
        model_path=FINETUNED_MODEL_PATH,
        model_title=(
            "GLOTLID v3 FINETUNED "
            "(5 epochs, LR=0.005)"
        ),
        output_prefix="glotlid_v3_finetuned"
    )
)


# ============================================================
# 10. FINAL ZERO-SHOT TABLE
# ============================================================

print("\n")
print("=" * 120)
print("ZERO-SHOT GLOTLID v3 — COPY THESE VALUES TO YOUR SPREADSHEET")
print("=" * 120)

print(
    zero_shot_table
    .fillna("N/A")
    .to_string()
)


# ============================================================
# 11. FINAL FINETUNED TABLE
# ============================================================

print("\n")
print("=" * 120)
print("FINETUNED GLOTLID v3 — COPY THESE VALUES TO YOUR SPREADSHEET")
print("=" * 120)

print(
    finetuned_table
    .fillna("N/A")
    .to_string()
)


# ============================================================
# 12. OVERALL RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("ZERO-SHOT OVERALL RESULTS")
print("=" * 80)

print(
    zero_shot_overall.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("FINETUNED OVERALL RESULTS")
print("=" * 80)

print(
    finetuned_overall.to_string(
        index=False
    )
)


# ============================================================
# 13. SAVE FINAL TABLES AS CSV
# ============================================================

zero_shot_summary_path = os.path.join(
    BENCHMARK_OUTPUT_DIR,
    "glotlid_v3_zero_shot_spreadsheet.csv"
)

finetuned_summary_path = os.path.join(
    BENCHMARK_OUTPUT_DIR,
    "glotlid_v3_finetuned_spreadsheet.csv"
)


zero_shot_table.to_csv(
    zero_shot_summary_path
)

finetuned_table.to_csv(
    finetuned_summary_path
)


print("\nSaved final spreadsheet tables:")

print(
    zero_shot_summary_path
)

print(
    finetuned_summary_path
)


# ============================================================
# 14. TAB-SEPARATED VERSION FOR EASY COPY/PASTE
# ============================================================

print("\n")
print("=" * 120)
print("ZERO-SHOT — TAB-SEPARATED")
print("=" * 120)

print(
    zero_shot_table
    .fillna("N/A")
    .to_csv(
        sep="\t"
    )
)


print("\n")
print("=" * 120)
print("FINETUNED — TAB-SEPARATED")
print("=" * 120)

print(
    finetuned_table
    .fillna("N/A")
    .to_csv(
        sep="\t"
    )
)

BENCHMARK FILES
commonlid.jsonl
flores_plus.jsonl
wili-2018.jsonl

Loaded benchmark rows:
commonlid: 77974 rows
flores_plus: 16155 rows
wili-2018: 14047 rows


####################################################################################################
GLOTLID v3 ZERO-SHOT
####################################################################################################

Loading model:
models/finetuned/GlotLID_v3/glotlid_v3_original.bin
Number of model labels: 2102

Target label support in model:
__label__sin_Sinh         True
__label__pli_Sinh         False
__label__san_Sinh         False
__label__san_Deva         True


GLOTLID v3 ZERO-SHOT - commonlid
Samples: 77974

True-label counts:
Sinhala-Sinh      : 2693
Pali-Sinh         : 3027
Sanskrit-Sinh     : 1327
Sanskrit-Deva     : 895
English-Latn      : 27461
Tamil-Taml        : 81
Hindi-Deva        : 3666
Bengali-Beng      : 1886
Arabic-Arab       : 26152
French-Latn       : 3233
German-Latn       : 7553

-----------------